# Lecture: How to Benchmark GPU Ops (Bandwidth, FLOPS, Roofline)

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Define bandwidth (GB/s), throughput (GFLOP/s), and arithmetic intensity (flops per byte) with the lab's four ops as examples (§4: vecadd 0.083, matmul-1024 170.7).
- Classify an op as memory-bound or compute-bound from its intensity against the ridge 6.67 (§5–§6) — and say which ceiling it should chase.
- List the three rules of honest GPU timing (warmup, `synchronize()`, median over repeats, §7) and why small sizes measure overhead, not speed (§8).

## The big idea

Every operation is either thirsty or hungry. A **memory-bound** op (like adding two vectors: 1 flop per 12 bytes = 0.083 flop/byte) does barely one unit of math per byte it drinks — its speed limit is **bandwidth**, how fast bytes flow (GB/s), like a cocktail straw limiting your sipping. A **compute-bound** op (like 1024×1024 matrix multiplication, intensity 170.7 flop/byte, which reuses each byte hundreds of times) is limited instead by raw math rate (**FLOPS**, floating-point operations per second), like a kitchen limited by chefs, not deliveries. The decider is one division: **arithmetic intensity = flops ÷ bytes moved** — and the four lab ops spread 2048x apart on that single scale. Sketching both ceilings on one chart is called the **roofline model** — you always live under the lower of the two roofs, which cross at the **ridge point** (6.67 here). The lab's twist: timing must be honest. GPUs work asynchronously, first runs include setup costs (so **warm up**), one lucky run means nothing (so take the **median** of repeats), and at tiny sizes you only measure fixed **overhead** (~microseconds of launch cost plus **H2D transfer** over PCIe), which is why the CPU 'wins' there — vendor libraries (**BLAS/cuBLAS**) behind NumPy/torch are the ceiling your custom kernel chases later.

In [ ]:
import numpy as np  # bring in numpy for the bytes-vs-ops arithmetic
import matplotlib.pyplot as plt  # bring in pyplot for the worked-example bars
np.random.seed(5)  # fix the seed (models below are fixed; seed keeps the habit)
n = 1_000_000  # define worked size: 1M elements of vecadd (12 MB of traffic)
vec_bytes = 3 * n * 4  # vecadd bytes: 2 reads + 1 write of 4-byte floats = 12 bytes/element
vec_flops = 1 * n  # vecadd flops: 1 add per element
print('vecadd 1M: bytes =', vec_bytes, 'flops =', vec_flops)  # print 12000000 bytes vs 1000000 flops
print('vecadd intensity:', round(vec_flops / vec_bytes, 3), 'flop/byte (thirsty)')  # print 0.083: drinks far more than it chews
m = 1024  # define worked matmul size: 1024x1024 (the lab's largest rung)
mm_flops = 2 * m ** 3  # matmul flops: 2*m^3 multiply-adds = 2147483648
mm_bytes = 3 * m * m * 4  # matmul bytes: 3 matrices of 4-byte floats = 12582912
print('matmul 1024: flops =', mm_flops, 'bytes =', mm_bytes)  # print the big-flop, small-byte pair
print('matmul intensity:', round(mm_flops / mm_bytes, 1), 'flop/byte (hungry)')  # print 170.7: chews each byte ~171 times
print('spread:', round((mm_flops / mm_bytes) / (vec_flops / vec_bytes), 0), 'x apart on one scale')  # print 2048.0: the range
labels = ['vecadd bytes\n(MB)', 'vecadd flops\n(M)', 'matmul bytes\n(MB)', 'matmul flops\n(G)']  # four bar labels
vals = [vec_bytes / 1e6, vec_flops / 1e6, mm_bytes / 1e6, mm_flops / 1e9]  # four quantities in matching units
fig, ax = plt.subplots(figsize=(7, 4))  # create one figure with one axes
ax.bar(labels, vals, color=['steelblue', 'darkorange', 'steelblue', 'darkorange'])  # draw bytes-blue, flops-orange bars
ax.set_ylabel('MB or M/G ops (same bar height scale)')  # label y as the worked-example quantities
ax.set_title('Anchor: vecadd moves 12 MB for 1 M flops; matmul 12.6 MB for 2.1 G flops')  # title the thirst/hunger contrast
ax.grid(True, axis='y', alpha=0.3)  # faint horizontal grid for readability
fig.tight_layout()  # fix spacing so labels fit
plt.show()  # display the figure

## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. §4 counts bytes and ops, §5 draws the two roofs, §6 seats each op under its roof, §7–§8 keep the stopwatch honest.

| Term | One-liner | Section |
|---|---|---|
| bandwidth (GB/s) | Bytes moved per second — the speed limit for thirsty ops | §4 |
| FLOPS / GFLOP/s | Floating-point ops per second (G = billions) — the limit for hungry ops | §4 |
| arithmetic intensity | Flops ÷ bytes moved (vecadd 0.083, reduce 0.25, softmax 0.312, matmul-1024 170.7) | §4 |
| roofline model | Attainable speed vs intensity under two ceilings; you live under the lower roof | §5 |
| memory-bound | Low intensity (below ridge 6.67): runtime set by bandwidth — move fewer bytes | §6 |
| compute-bound | High intensity (above ridge): runtime set by math rate — raise flop throughput | §6 |
| warmup / synchronize / median | Honest-timing trio: absorb setup, wait for async work, resist outliers | §7 |
| overhead / launch cost | Fixed microseconds per kernel launch — dominates tiny sizes | §8 |
| H2D transfer | Host-to-device copy over PCIe; timed separately from resident compute | §8 |
| BLAS / cuBLAS | Vendor-tuned math libraries behind NumPy/torch matmul — the ceiling to chase | §8 |

### §4 Bandwidth, FLOPS, and intensity — counting bytes vs ops

**In one sentence: bandwidth is bytes moved per second (GB/s), FLOPS is math operations per second (GFLOP/s), and arithmetic intensity is flops divided by bytes — vecadd 0.083, reduction 0.25, softmax 0.312, matmul-1024 170.7.**

Worked examples (all FP32, 4 bytes per float): vector-add of length `n` reads two vectors and writes one — 12 bytes per element for 1 flop, so 0.083 flop/byte no matter how big `n` gets. Reduction reads once and barely writes — 4 bytes per element for 1 flop, so 0.25. Row-softmax makes about 4 passes over ~16 bytes per element for ~5 flops, so 0.312. But 1024×1024 matmul does 2.1 G flops on only 12.6 MB — 170.7 flop/byte, 2048x above vecadd — because its O(n³) math reuses each O(n²) byte O(n) times. That reuse gap is the entire game. Watch it: the four intensities as bars on a log scale against the ridge line.

In [ ]:
import numpy as np  # bring in numpy for the intensity arithmetic
import matplotlib.pyplot as plt  # bring in pyplot for the intensity bar chart
np.random.seed(6)  # fix the seed (no randomness below; seed keeps the habit)
ops = ['vecadd', 'reduce', 'softmax', 'matmul-1024']  # define the 4 lab ops in thirsty-to-hungry order
m = 1024  # define matmul size for the exact intensity math
vec_i = 1.0 / 12.0  # vecadd intensity: 1 flop per 12 bytes = 0.083
red_i = 1.0 / 4.0  # reduction intensity: 1 flop per 4 bytes = 0.25
soft_i = 5.0 / 16.0  # softmax intensity: ~5 flops per ~16 bytes = 0.312
mm_i = (2.0 * m ** 3) / (3.0 * m * m * 4.0)  # matmul intensity: exact 2m^3 / 12m^2 = 170.7
intensity = np.array([vec_i, red_i, soft_i, mm_i])  # collect the four flops-per-byte numbers
for name, val in zip(ops, intensity):  # loop over ops with their computed intensities
    print(name, 'intensity:', round(float(val), 3), 'flop/byte')  # print each op's flops-per-byte number
print('spread max/min:', round(float(intensity.max() / intensity.min()), 0), 'x apart')  # print 2048.0: the range
ridge = 200.0 / 30.0  # sketch ridge point: 200 GFLOP/s roof over 30 GB/s roof = 6.67 flop/byte
print('ridge:', round(float(ridge), 2), '(below = chase GB/s; above = chase GFLOP/s)')  # print the roofline rule
fig, ax = plt.subplots(figsize=(6, 4))  # create one figure with one axes
ax.bar(ops, intensity)  # draw one bar per op's intensity
ax.set_yscale('log')  # use log y because intensities span 0.083 to 170.7
ax.set_ylabel('arithmetic intensity, flop/byte (log scale)')  # label y as flops per byte
ax.set_title('Low bars chase bandwidth, the tall bar chases flops')  # plain-English title
ax.axhline(ridge, linestyle='--')  # draw a dashed ridge line separating the two regimes
ax.text(3.05, ridge * 1.4, 'ridge 6.67 (sketch)')  # annotate the dashed ridge line
ax.grid(True, axis='y', alpha=0.3)  # faint horizontal grid for readability
fig.tight_layout()  # fix spacing so labels fit
plt.show()  # display the figure

### §5 The roofline — the plot itself

**In one sentence: the roofline plots attainable speed against intensity under two ceilings — a slanted bandwidth roof (`bandwidth × intensity`) and a flat compute roof (peak GFLOP/s) — and every op lives under the lower of the two, crossing at the ridge 6.67.**

Read it left to right: at intensity 0.083 the bandwidth roof allows only 2.5 GFLOP/s (30 × 0.083), so vecadd can never touch the 200 GFLOP/s ceiling no matter how clever the code. As intensity grows the slanted roof rises until it hits the flat roof at the ridge — beyond that (matmul at 170.7) only more flops help. Optimizations *move* points: fewer bytes slide a point right (higher intensity), faster math pushes it up. The sketch below uses CPU-like ceilings (30 GB/s, 200 GFLOP/s); real GPU roofs sit higher but the shape never changes. Watch it: the two roofs plus the four lab ops pinned under them.

In [ ]:
import numpy as np  # bring in numpy for the roofline curve math
import matplotlib.pyplot as plt  # bring in pyplot for the roofline plot
np.random.seed(6)  # fix the seed (no randomness below; seed keeps the habit)
bw, peak = 30.0, 200.0  # sketch ceilings: 30 GB/s bandwidth roof, 200 GFLOP/s compute roof
ridge = peak / bw  # ridge intensity where roofs cross: 200/30 = 6.67 flop/byte
xs = np.logspace(-2, 3, 200)  # 200 intensity points from 0.01 to 1000 flop/byte
roof = np.minimum(bw * xs, peak)  # attainable speed: lower of the slanted and flat roofs
pts = {'vecadd': 1.0/12.0, 'reduce': 0.25, 'softmax': 5.0/16.0, 'matmul-1024': (2*1024**3)/(3*1024*1024*4)}  # four lab intensities
print('ridge at intensity:', round(float(ridge), 2))  # print 6.67: where the roofs cross
for name, it in pts.items():  # loop over the four lab ops
    print(name, 'attainable:', round(float(min(bw*it, peak)), 2), 'GFLOP/s')  # print each op's ceiling (2.5 .. 200)
fig, ax = plt.subplots(figsize=(7, 4.5))  # create one figure with one axes
ax.loglog(xs, roof, 'k-', label='roofline (lower of two roofs)')  # draw the bent roofline curve
ax.loglog(xs, bw * xs, '--', alpha=0.4, label='bandwidth roof (30 x intensity)')  # draw the slanted roof extended
ax.axhline(peak, linestyle=':', alpha=0.5, label='compute roof (200 GFLOP/s)')  # draw the flat roof extended
for name, it in pts.items():  # pin each lab op under its roof
    ax.plot(it, min(bw*it, peak), 'ro', ms=7)  # mark the op's attainable point
    ax.text(it*1.15, min(bw*it, peak)*1.3, name, fontsize=9)  # label the point with the op name
ax.set_xlabel('arithmetic intensity, flop/byte (log)')  # label x as flops per byte
ax.set_ylabel('attainable GFLOP/s (log)')  # label y as achievable speed
ax.set_title('Roofline: thirsty ops hug the slope (2.5), matmul hits the ceiling (200)')  # title the reading rule
ax.legend(fontsize=8)  # show which line is which roof
ax.grid(True, which='both', alpha=0.3)  # faint grid on both log scales
fig.tight_layout()  # fix spacing so labels fit
plt.show()  # display the figure

### §6 Memory-bound vs compute-bound — points on the roofline

**In one sentence: below the ridge 6.67 an op is memory-bound (its runtime is set by bandwidth — optimize by moving fewer bytes: fewer passes, narrower dtypes), far above it an op is compute-bound (runtime set by math rate — optimize with Tensor Cores and better kernels).**

The verdicts: vecadd (0.083), reduction (0.25), and softmax (0.312) all sit on the slanted roof — their attainable speeds are 2.5, 7.5, and 9.38 GFLOP/s, capped by the 30 GB/s straw no matter the flop ceiling. Matmul-1024 (170.7) sits on the flat roof at 200 GFLOP/s — giving it more bandwidth changes nothing; only faster math helps. Near the ridge either ceiling can bind, so measure. Watch it: the attainable-speed bars showing three ops capped low by bandwidth and one capped high by flops.

In [ ]:
import numpy as np  # reuse numpy for the attainable-speed math
import matplotlib.pyplot as plt  # reuse pyplot for the verdict bars
np.random.seed(6)  # fix the seed (no randomness below; seed keeps the habit)
bw, peak = 30.0, 200.0  # reuse the sketch ceilings: 30 GB/s and 200 GFLOP/s
names = ['vecadd', 'reduce', 'softmax', 'matmul-1024']  # four lab ops in roofline order
its = np.array([1.0/12.0, 0.25, 5.0/16.0, (2*1024**3)/(3*1024*1024*4)])  # four intensities: 0.083 .. 170.7
att = np.minimum(bw * its, peak)  # attainable GFLOP/s per op: min of the two roofs
for name, a in zip(names, att):  # loop over ops with attainable speeds
    verdict = 'MEMORY-bound (chase GB/s)' if a < peak else 'COMPUTE-bound (chase GFLOP/s)'  # classify by binding roof
    print(name, round(float(a), 2), 'GFLOP/s ->', verdict)  # print each verdict: 2.5, 7.5, 9.38, 200
fig, ax = plt.subplots(figsize=(7, 4))  # create one figure with one axes
colors = ['steelblue', 'steelblue', 'steelblue', 'darkorange']  # blue = bandwidth-capped, orange = flop-capped
ax.bar(names, att, color=colors)  # draw one attainable-speed bar per op
ax.set_ylabel('attainable GFLOP/s (higher is better)')  # label y as achievable speed
ax.set_title('Verdicts: three ops capped by bandwidth, matmul capped by flops')  # title the classification
ax.grid(True, axis='y', alpha=0.3)  # faint horizontal grid for readability
fig.tight_layout()  # fix spacing so labels fit
plt.show()  # display the figure

### §7 Warmup, synchronize, and median — honest timing

**In one sentence: warmup runs absorb first-iteration setup and autotuning, `synchronize()` blocks until async GPU work actually finishes so the timer includes real work, and the median over repeats resists one lucky or unlucky run.**

The strip below times one op 12 times: the first two runs are slow (cold caches, kernel autotune — 3.2 and 2.1 ms vs a 1.0 ms steady state), run 7 is a noise spike (2.8 ms from a background hiccup), and the remaining nine cluster at 1.0 ms. The mean (1.426 ms) is dragged up by all three outliers; the median (1.011 ms) reports the steady state — printed below. Rule of thumb from the lab: `warmup=3, repeat=10`, median reported, sync before *and* after the timed region. Watch it: the strip of 12 runs with cold, spike, mean, and median marked.

In [ ]:
import numpy as np  # reuse numpy for the median/mean math
import matplotlib.pyplot as plt  # reuse pyplot for the noisy-timing strip plot
np.random.seed(7)  # fix the seed (jitter below is reproducible)
steady = 1.0 + np.random.randn(9) * 0.05  # nine steady-state runs near 1.0 ms (small jitter)
runs = np.array([3.2, 2.1] + [round(float(v), 3) for v in steady[:5]] + [2.8] + [round(float(v), 3) for v in steady[5:]])  # 12 runs: 2 cold + 5 steady + spike + 4 steady
print('12 runs (ms):', np.round(runs, 2).tolist())  # print all twelve timings: cold 3.2/2.1, spike 2.8, rest ~1.0
print('mean:', round(float(runs.mean()), 3), 'ms (dragged up by cold + spike)')  # print 1.426: the fragile average
print('median:', round(float(np.median(runs)), 3), 'ms (steady state)')  # print ~1.0: the robust report
print('rule: warmup=3 absorbs cold, sync brackets async work, median resists spikes')  # print the trio rule
fig, ax = plt.subplots(figsize=(8, 3.2))  # make a wide flat figure for the strip
ax.plot(runs, 'o', ms=7, label='measured runs')  # draw one dot per timed run
ax.axhline(runs.mean(), color='red', linestyle='--', label='mean (fragile)')  # draw the outlier-dragged mean
ax.axhline(np.median(runs), color='green', linestyle='-', label='median (reported)')  # draw the robust median
ax.text(0, 3.2, 'cold (no warmup)', fontsize=8, color='red')  # annotate the slow first runs
ax.text(7, 2.85, 'spike (noise)', fontsize=8, color='red')  # annotate the background hiccup
ax.set_xlabel('repeat number')  # label x as the repetition index
ax.set_ylabel('milliseconds (lower is better)')  # label y as per-run time
ax.set_title('Noisy timing: warmup kills cold runs, median kills spikes')  # title the honesty trio
ax.legend(fontsize=8)  # show which line is mean vs median
ax.grid(True, axis='y', alpha=0.3)  # faint horizontal grid for readability
fig.tight_layout()  # fix spacing so labels fit
plt.show()  # display the figure

### §8 Overhead, H2D, and BLAS — what small-n really measures

**In one sentence: every kernel launch pays fixed overhead (~2 µs) plus any host-to-device copy, so at small sizes (10k elements ≈ 0.12 MB) the ~microsecond fee dwarfs the byte-moving time and timings measure overhead, not bandwidth — while vendor BLAS/cuBLAS libraries behind NumPy/torch matmul define the speed ceiling your code chases.**

The bars below model vecadd time as overhead + bytes/bandwidth: at n = 10k the 2 µs fee is most of the total (bandwidth barely matters — any backend reports similar microseconds); at n = 4M the 48 MB of traffic dominates and true GB/s emerges. The lab therefore sweeps sizes (10k → 4M) and keeps H2D copies *outside* the compute timer, treating transfer as a separate what-if: including it answers a different question (end-to-end cost) than resident compute (kernel quality). BLAS note: NumPy/torch matmul call hand-tuned assembly (MKL/OpenBLAS/cuBLAS) — beating them with a naive triple loop is not the goal; matching their structure (Project 6's tiling) is. Watch it: the overhead-vs-traffic bars at two sizes.

In [ ]:
import numpy as np  # reuse numpy for the overhead-vs-traffic model
import matplotlib.pyplot as plt  # reuse pyplot for the small-n bars
np.random.seed(5)  # fix the seed (model below is fixed; seed keeps the habit)
overhead = 2e-6  # fixed per-launch overhead in seconds (2 microseconds, paid at every size)
bw = 30e9  # assumed memory bandwidth: 30 GB/s (a CPU-like ceiling for the sketch)
ns = np.array([10_000, 4_000_000])  # two sizes: 10k (overhead-bound) vs 4M (bandwidth-bound)
traffic = 12.0 * ns  # vecadd bytes per size: 12 bytes per element
t_total = overhead + traffic / bw  # modeled time: fixed fee plus byte-moving time
print('n=10k: overhead 2.0 us vs traffic', round(float(traffic[0]/bw*1e6), 2), 'us -> overhead dominates')  # print 2 vs 4: fee matters
print('n=4M: overhead 2.0 us vs traffic', round(float(traffic[1]/bw*1e3), 2), 'ms -> bandwidth dominates')  # print 2us vs 1.6ms
print('small-n lesson: microseconds measure launch + H2D, not GB/s')  # print the takeaway
print('BLAS note: numpy/torch matmul = tuned MKL/cuBLAS (your ceiling, not your baseline)')  # print the ceiling note
fig, ax = plt.subplots(figsize=(7, 4))  # create one figure with one axes
x = np.arange(2)  # two x positions: small-n and large-n
ax.bar(x, [overhead*1e6]*2, 0.5, label='overhead (2 us always)')  # draw the fixed-fee slice at both sizes
ax.bar(x, traffic / bw * 1e6, 0.5, bottom=[overhead*1e6]*2, label='traffic time (grows with n)')  # stack traffic on top
ax.set_xticks(x)  # place ticks at the two sizes
ax.set_xticklabels(['n=10k (overhead-bound)', 'n=4M (bandwidth-bound)'])  # name each regime
ax.set_ylabel('modeled microseconds (log)')  # label y as total modeled time
ax.set_yscale('log')  # log y: totals span ~6 us to ~1600 us
ax.set_title('Small-n bars: the fee dominates left, traffic dominates right')  # title the regime split
ax.legend(fontsize=8)  # show fee vs traffic colors
ax.grid(True, axis='y', alpha=0.3)  # faint horizontal grid for readability
fig.tight_layout()  # fix spacing so labels fit
plt.show()  # display the figure

## How it all fits together — the system map §9

One benchmark point is a pipeline with every § term playing a position. **Count** (§4): flops and bytes give intensity (vecadd 0.083). **Roofline** (§5–§6): intensity picks the binding roof (bandwidth, attainable 2.5 GFLOP/s). **Time honestly** (§7): warmup, then a synced timed loop, then the median. **Sanity-check size** (§8): tiny-n microseconds mean overhead/H2D, not bandwidth; compare against BLAS ceilings, not naive loops. The map below pins each term to its station; the numbers walk vecadd at n = 1M from intensity to its roofline position and predicted time.

In [ ]:
import matplotlib.pyplot as plt  # only pyplot: this cell is a pure diagram
fig, ax = plt.subplots(figsize=(11, 3.8))  # wide figure for the benchmarking pipeline
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 14)  # fix placement range
ax.set_ylim(0, 4)  # fix placement range
stages = [('warmup\n§7', 1.2), ('timed loop\n§7', 3.8), ('sync\n§7', 6.2), ('median\n§7', 8.6), ('roofline pt\n§5+§6', 11.0), ('size check\n§8', 13.0)]  # pipeline stations in order
for name, xc in stages:  # draw each station box
    ax.text(xc, 2.5, name, ha='center', va='center', fontsize=8,  # centered station label
            bbox=dict(boxstyle='round', facecolor='lightblue'))  # blue = measurement pipeline
for xa, xb in [(1.8, 3.0), (4.6, 5.5), (6.8, 7.9), (9.2, 10.2), (11.8, 12.4)]:  # gaps between stations
    ax.annotate('', xy=(xb, 2.5), xytext=(xa, 2.5), arrowprops=dict(arrowstyle='->'))  # forward arrows left to right
ax.text(7, 3.6, 'ONE BENCHMARK POINT: warm up, time synced loops, median, seat on roofline', ha='center', fontsize=9, fontweight='bold')  # phase banner
ax.text(7, 0.5, 'intensity 0.083 -> bandwidth roof (2.5 GFLOP/s); tiny-n us = overhead, not speed', ha='center', fontsize=9, fontweight='bold', color='darkred')  # red = the two reading rules
ax.set_title('System map §9: warmup → timed loop → sync → median → roofline point')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the map

In [ ]:
import numpy as np  # reuse numpy for the one-op roofline trace
np.random.seed(9)  # fix the seed (numbers below are fixed; habit kept)
n = 1_000_000  # trace size: 1M-element vecadd (12 MB traffic, bandwidth-bound)
flops, nbytes = 1.0 * n, 12.0 * n  # COUNT (§4): 1M flops, 12M bytes
intensity = flops / nbytes  # INTENSITY: 0.083 flop/byte
bw, peak = 30.0, 200.0  # sketch ceilings in GB/s and GFLOP/s (§5)
attainable = min(bw * intensity, peak)  # ROOFLINE (§5+§6): min(2.5, 200) = 2.5 GFLOP/s, memory-bound
t_pred = flops / (attainable * 1e9)  # predicted time = flops / attainable rate = 0.4 ms
print('intensity:', round(float(intensity), 3), 'flop/byte -> MEMORY-bound (below ridge 6.67)')  # print 0.083 verdict
print('attainable:', round(float(attainable), 2), 'GFLOP/s (bandwidth roof 30 x 0.083)')  # print 2.5: the binding roof
print('predicted time for 1M vecadd:', round(float(t_pred*1e3), 2), 'ms')  # print 0.4: the roofline prediction
print('honest check: warmup + sync + median (§7); at 10k this would be overhead µs (§8)')  # print the timing footnote

## Bridge to the lab

Open `notebook.ipynb` now. It turns the sketch above into a real 4-op × 3-backend measurement:

1. **Cell 1 (goal markdown)** — the matrix: NumPy-CPU vs torch-CPU vs torch-GPU across vector-add, matmul, reduction, softmax. You can already seat each op (§6).
2. **Cell 2 (setup + `time_op`)** — seeds, env logging, and the honest-timing trio: warmup + `synchronize()` + median (§7). This cell is the vocabulary table in code.
3. **Cell 3 (op definitions + checks)** — NumPy and torch versions of all four ops, then `assert allclose` proofs they compute the same answers (benchmarks must compare equal math, §4's counts assume it).
4. **Cell 4 (vecadd + reduce sweep)** — the memory-bound rungs (§6): bandwidth math is bytes÷time (12 bytes/element for vecadd, 4 for reduce). Watch small-n rows report microseconds of overhead (§8).
5. **Cell 5 (matmul + softmax sweep)** — the compute-bound rung (flops = 2m³, GFLOP/s = flops÷time, §4) plus multi-pass softmax (0.312, §6). GPU rows auto-skip without CUDA.
6. **Cell 6 (CSV + plots)** — `results/bench.csv`, the log-log runtime plot and the GB/s vs GFLOP/s throughput panels (the §5 roofs, real).
7. **Cell 7 (roofline reading)** — intensity math per op (§4: 0.083 → 170.7, 2048x spread), best-bandwidth and best-matmul anchors vs hardware ceilings, the fp32→fp64 what-if (2× bytes ⇒ ~2× time on memory-bound ops), and the pointer to Project 5.
8. **Cell 8 (README prompts)** — the one-liner to memorize: `intensity = flops / bytes` decides everything (§9).

## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **An op moves 12 MB and does 1 M flops. Memory- or compute-bound, and what should it chase?**

*Answer: intensity = 1/12 ≈ 0.083 flop/byte (§4) — far below the ridge 6.67, so memory-bound (§6). Chase GB/s: fewer passes, narrower dtype, resident data. The §9 trace predicts 0.4 ms at 2.5 GFLOP/s on the sketch roofs.*

2. **Why do honest GPU benchmarks need warmup, `synchronize()`, and medians?**

*Answer: warmup absorbs first-run setup and autotuning (the 3.2/2.1 ms cold runs, §7);* `synchronize()` *waits for asynchronous kernels so the timer includes real work; the median (1.011 ms) resists one lucky or unlucky spike that drags the mean to 1.426.*

3. **Two backends report peak bandwidth at n=10k and at n=4M. Which number reflects true DRAM bandwidth, and why?**

*Answer: the 4M number (§8). At 10k the 2 µs launch fee plus any H2D copy dominate the ~4 µs traffic time, so microseconds measure overhead — and small data may sit in cache. Bandwidth claims need streaming sizes, and BLAS/cuBLAS ceilings (§8) are the fair comparison, not naive loops.*